# Exploration des colonnes candidates au partitionnement

Cette exploration permet d’identifier les colonnes pouvant servir de clés de partitionnement. Les couleurs constituent une première indication : la décision finale dépendra également des usages, du nombre de fichiers, du coût et des performances des requêtes.

## Métriques calculées

Pour chaque table :

$$
\text{rows} = \text{nombre total de lignes}
$$

Pour chaque colonne :

$$
\text{non-null values}
=
\text{rows} - \text{null count}
$$

$$
\text{unique count}
=
\text{nombre de valeurs distinctes non nulles}
$$

$$
\text{unique ratio}
=
\frac{\text{unique count}}
{\text{non-null values}}
$$

Le ratio d’unicité estime la proportion de partitions distinctes qui pourraient être créées si la colonne était utilisée comme clé.

## Règles appliquées

### Taille de la table

| Condition | Couleur | Interprétation |
|---|---|---|
| `rows < 20` | Jaune | Table très petite : le partitionnement est probablement inutile |
| `rows >= 20` | Aucune | La table peut être étudiée normalement |

### Valeurs manquantes

| Condition | Couleur | Interprétation |
|---|---|---|
| `null_count = 0` | Aucune | La colonne peut être étudiée |
| `null_count > 0` | Rouge | Colonne risquée pour le partitionnement |

Une colonne contenant des valeurs manquantes est écartée de la présélection afin d’éviter des partitions nulles ou ambiguës.

### Cardinalité

| Ratio de valeurs uniques | Couleur | Interprétation |
|---|---|---|
| `< 5 %` | Rouge | Trop peu de groupes : partitionnement probablement trop grossier |
| `5 % – 40 %` | Vert | Candidat potentiellement intéressant |
| `> 40 % – 75 %` | Jaune | À étudier selon les requêtes et le volume |
| `> 75 %` | Rouge | Risque de surpartitionnement et de nombreux petits fichiers |

## Interprétation

Une bonne clé de partitionnement doit :

- correspondre à un filtre fréquent dans les requêtes ;
- créer des groupes suffisamment importants ;
- réduire les octets lus par le moteur de requête ;
- éviter la création d’un trop grand nombre de petits fichiers ;
- ne contenir aucune valeur manquante dans le cadre de cet audit.

Les colonnes rouges ne sont pas automatiquement interdites. Elles sont simplement considérées comme peu favorables et ne doivent être retenues que si un besoin métier précis le justifie.

## Décision finale

Les candidats retenus seront comparés selon :

- le nombre de partitions ;
- le nombre d’objets créés ;
- la taille moyenne des fichiers ;
- le temps d’écriture et d’envoi ;
- le temps d’exécution des requêtes ;
- les octets traités par BigQuery ;
- le coût de stockage ;
- le coût des opérations ;
- le coût des requêtes.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

SOURCE_DIR = Path("../../legacy_csv")


def explore_csv(csv_path):
    df = pd.read_csv(csv_path)
    total_rows = len(df)

    results = []

    for column in df.columns:
        non_null = df[column].notna().sum()
        unique_count = df[column].nunique(dropna=True)

        unique_ratio = (
            unique_count / non_null
            if non_null > 0
            else 0
        )

        results.append({
            "table": csv_path.stem,
            "column": column,
            "dtype": str(df[column].dtype),
            "rows": total_rows,
            "non_null": non_null,
            "null_count": df[column].isna().sum(),
            "unique_count": unique_count,
            "unique_ratio": unique_ratio,
            "sample_values": df[column].dropna().unique()[:2].tolist(),
        })

    return pd.DataFrame(results)

def color_unique_ratio(value):
    if value < 0.05:
        # Vert foncé : faible cardinalité
        return "background-color: #2d6a4f; color: #ffffff"
    elif value <= 0.40:
        # Vert clair : cardinalité modérée
        return "background-color: #b7e4c7; color: #081c15"
    elif value <= 0.75:
        # Jaune : à étudier
        return "background-color: #ffe699; color: #332701"
    else:
        # Rouge : forte cardinalité
        return "background-color: #ffadad; color: #450920"

def color_rows(value):
    if value < 20:
        return "background-color: #ffe699; color: #332701"
    return ""

def color_null_count(value):
    if value > 0:
        return "background-color: #ffadad; color: #450920"
    return ""

def color_unique_count(value):
    if value > 1000:
        return "background-color: #ff4d4d; color: #ffffff"
    return ""

highlight_columns = {
    "background-color": "#e3f2fd",
    "border-left": "2px solid #1976d2",
    "border-right": "2px solid #1976d2",
    "font-weight": "bold",
}

all_results = []

for csv_path in sorted(SOURCE_DIR.glob("*.csv")):
    result = explore_csv(csv_path)
    all_results.append(result)

    print(f"\n{csv_path.name}")

    display(
        result.style
        
        .set_properties(
            subset=["rows", "null_count", "unique_count"],
            **highlight_columns
        )

        
        .set_table_styles([
            {
                "selector": "th.col_heading.level0.col3",
                "props": list(highlight_columns.items()),
            },
            {
                "selector": "th.col_heading.level0.col5",
                "props": list(highlight_columns.items()),
            },
            {
                "selector": "th.col_heading.level0.col6",
                "props": list(highlight_columns.items()),
            },
        ])
        .applymap(
            color_unique_ratio,
            subset=["unique_ratio"]
        )
        .applymap(
            color_null_count,
            subset=["null_count"]
        )
        .applymap(
            color_rows,
            subset=["rows"]
        )
        .applymap(
            color_unique_count,
            subset=["unique_count"]
        )

        .format({
            "unique_ratio": "{:.1%}",
        })
    )

audit_df = pd.concat(all_results, ignore_index=True)


buyer.csv


,table,column,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,buyer,Unnamed: 0,int64,100000,100000,0,100000,100.0%,"[0, 1]"
1,buyer,buyer_id,object,100000,100000,0,100000,100.0%,"['AE222BBOVZIF42YOOPNBXL4UUMYA', 'AE222FP7YRNFCEQ2W3ZDIGMSYTLQ']"



carrier.csv


,table,column,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,carrier,Unnamed: 0,int64,4,4,0,4,100.0%,"[0, 1]"
1,carrier,carrier_id,int64,4,4,0,4,100.0%,"[1, 2]"
2,carrier,carrier_name,object,4,4,0,4,100.0%,"['Albatros Parcels', 'Blue Heron Logistics']"



cart.csv


...

,table,column,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,shipping_details,Unnamed: 0,int64,200001,200001,0,200001,100.0%,"[0, 1]"
1,shipping_details,address_id,int64,200001,200001,0,200001,100.0%,"[1, 2]"
2,shipping_details,street_address,object,200001,200001,0,195247,97.6%,"['Lene-Fechner-Ring 32', '73, avenue William Valentin']"
3,shipping_details,city,object,200001,200001,0,54049,27.0%,"['Süßen', 'Allainville']"
4,shipping_details,state,object,200001,200001,0,83,0.0%,"['Baden-Württemberg', 'Île-de-France']"
5,shipping_details,zip,object,200001,200001,0,38417,19.2%,"['73079', '78660']"
6,shipping_details,country,object,200001,200001,0,6,0.0%,"['Germany', 'France']"
7,shipping_details,phone,int64,200001,200001,0,99998,50.0%,"[15737994260, 744501207]"



subscription.csv


,table,column,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,subscription,Unnamed: 0,int64,50091,50091,0,50091,100.0%,"[0, 1]"
1,subscription,subscription_id,int64,50091,50091,0,50091,100.0%,"[1, 2]"
2,subscription,c_id,object,50091,50091,0,50091,100.0%,"['AE222Y4WTST6BUZ4J5Y2H6QMBITQ', 'AE22355IAQGYY4EPWWVYX25J7AHA']"
3,subscription,start_date,object,50091,50091,0,366,0.7%,"['2025-03-02', '2025-02-25']"
4,subscription,end_date,object,50091,50091,0,366,0.7%,"['2026-03-02', '2026-02-25']"



wishlist_item.csv


,table,column,dtype,rows,non_null,null_count,unique_count,unique_ratio,sample_values
0,wishlist_item,Unnamed: 0,int64,24810,24810,0,24810,100.0%,"[0, 1]"
1,wishlist_item,product_id,object,24810,24810,0,18842,75.9%,"['0816091846', '0985393300']"
2,wishlist_item,buyer_id,object,24810,24810,0,19823,79.9%,"['AEIY4E2ZRDXCETPZJCOIP6QUS2GQ', 'AEKXN4RB2KFNZHKC3K6ETHDA3MCA']"


# Les meilleurs candidats temporels sont **orders.order_date, daily_deals.deal_date et subscription.start_date.**



| Priorité | Table produite | Partition proposée |
|---|---|---|
| Forte | `order_items` + `orders` | `order_month` |
| Forte | `order_items` + `orders` + `product` | `order_month`, `category_id` |
| Moyenne | `daily_deals` + `product` | `deal_month`, `category_id` |
| Moyenne | `shipment` + `product` | `delivery_month`, `category_id` |
| Faible | `payment` + `orders` | `order_month`, `method` |
| Faible | `seller_products` + `seller` + `product` | `seller_type`, `category_id` |

## Set up audit environment 

#### Using google cloud sdk for test form notebook

In [2]:
from google.cloud import storage, bigquery
import google.auth

PROJECT_ID = "data-engineering-a"

credentials, detected_project = google.auth.default()

print("Authentification ADC : OK")
print("Projet détecté :", detected_project)

storage_client = storage.Client(
    project=PROJECT_ID,
    credentials=credentials
)

bigquery_client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

print("Client Cloud Storage : OK")
print("Client BigQuery : OK")

Authentification ADC : OK
Projet détecté : data-engineering-a
Client Cloud Storage : OK
Client BigQuery : OK


#### Config GCP 

In [3]:
from pathlib import Path
import json
import pandas as pd
from google.cloud import storage, bigquery

PROJECT_ID = "data-engineering-a"
BUCKET_NAME = "data-engineering-a-alexandro"

AUDIT_PREFIX = "audit/partition_tests"
BRONZE_PREFIX = "bronze"

BIGQUERY_DATASET = "bronze_audit_alexandro"
LOCATION = "europe-west1"

SOURCE_DIR = Path("../../legacy_csv")
CONFIG_PATH = Path("explore_data_mapping.json")
LOCAL_AUDIT_DIR = Path("partition_audit_output")

storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

bucket = storage_client.get_bucket(BUCKET_NAME)

print("Bucket :", bucket.name)
print("Région :", bucket.location)
print("URI audit :", f"gs://{BUCKET_NAME}/{AUDIT_PREFIX}/")

Bucket : data-engineering-a-alexandro
Région : EUROPE-WEST1
URI audit : gs://data-engineering-a-alexandro/audit/partition_tests/


## What will be audited

In [4]:
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    partition_config = json.load(file)


table_count = len(partition_config)

layout_count = sum(
    len(candidates)
    for candidates in partition_config.values()
)

print("Tables sources :", table_count)
print("Layouts à tester :", layout_count)

Tables sources : 27
Layouts à tester : 43


## Generate all parquet and layout for audit

In [13]:
from datetime import datetime
from time import perf_counter

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

LOCAL_AUDIT_DIR = Path("../ingestion/audit_layout").resolve()
RUN_DIR = LOCAL_AUDIT_DIR / RUN_ID

local_metrics = []

for table_name, candidates in partition_config.items():

    csv_path = SOURCE_DIR / f"{table_name}.csv"
    df = pd.read_csv(csv_path)

    print(f"\n{table_name} — {len(df):,} lignes")

    for partition_columns in candidates:

        layout_name = (
            "unpartitioned"
            if partition_columns is None
            else "by_" + "_then_".join(partition_columns)
        )

        output_dir = RUN_DIR / table_name / layout_name
        output_dir.mkdir(parents=True, exist_ok=True)

        start = perf_counter()

        if partition_columns is None:

            df.to_parquet(
                output_dir / "data.parquet",
                engine="pyarrow",
                compression="snappy",
                index=False,
            )

            partition_count = 1

        else:

            df.to_parquet(
                output_dir,
                engine="pyarrow",
                compression="snappy",
                index=False,
                partition_cols=partition_columns,
                basename_template="part-{i}.parquet",
                max_partitions=20000,
            )

            partition_count = (
                df[partition_columns]
                .drop_duplicates()
                .shape[0]
            )

        write_seconds = perf_counter() - start

        parquet_files = list(
            output_dir.rglob("*.parquet")
        )

        object_count = len(parquet_files)

        total_bytes = sum(
            file.stat().st_size
            for file in parquet_files
        )

        average_file_bytes = (
            total_bytes / object_count
            if object_count > 0
            else 0
        )

        local_metrics.append({
            "run_id": RUN_ID,
            "table": table_name,
            "layout": layout_name,
            "partition_columns": (
                None
                if partition_columns is None
                else ", ".join(partition_columns)
            ),
            "rows": len(df),
            "partition_count": partition_count,
            "object_count": object_count,
            "total_bytes": total_bytes,
            "average_file_bytes": average_file_bytes,
            "local_write_seconds": write_seconds,
            "local_path": str(output_dir),
        })

        print(
            f"  {layout_name}: "
            f"{object_count:,} fichiers | "
            f"{total_bytes:,} octets | "
            f"{write_seconds:.4f} s"
        )

local_metrics_df = pd.DataFrame(local_metrics)

print("\nRun terminé :", RUN_ID)
print("Layouts créés :", len(local_metrics_df))


buyer — 100,000 lignes
  unpartitioned: 1 fichiers | 3,252,070 octets | 0.1714 s

carrier — 4 lignes
  unpartitioned: 1 fichiers | 2,768 octets | 0.0050 s

cart — 0 lignes
  unpartitioned: 1 fichiers | 2,944 octets | 0.0025 s

cart_items — 0 lignes
  unpartitioned: 1 fichiers | 2,366 octets | 0.0031 s

category — 2 lignes
  unpartitioned: 1 fichiers | 3,221 octets | 0.0024 s
  by_category_id: 2 fichiers | 5,568 octets | 0.0372 s

customer — 130,766 lignes
  unpartitioned: 1 fichiers | 9,895,404 octets | 0.1764 s

customer_payment — 100,000 lignes
  unpartitioned: 1 fichiers | 3,866,376 octets | 0.0457 s

customer_shipping — 200,001 lignes
  unpartitioned: 1 fichiers | 7,573,765 octets | 0.1185 s
  by_is_default: 2 fichiers | 7,965,882 octets | 0.1038 s

daily_deals — 0 lignes
  unpartitioned: 1 fichiers | 2,433 octets | 0.0006 s

discount — 0 lignes
  unpartitioned: 1 fichiers | 3,011 octets | 0.0006 s

order_items — 408,560 lignes
  unpartitioned: 1 fichiers | 3,655,364 octets | 0.05

## Verify content created 

In [14]:
all_local_files = list(RUN_DIR.rglob("*.parquet"))

total_local_bytes = sum(
    file.stat().st_size
    for file in all_local_files
)

print("Run :", RUN_ID)
print("Dossier local :", RUN_DIR)
print("Layouts :", len(local_metrics_df))
print("Fichiers Parquet :", len(all_local_files))
print("Volume total :", f"{total_local_bytes / 1024**2:.2f} MiB")
print(
    "Destination :",
    f"gs://{BUCKET_NAME}/{AUDIT_PREFIX}/{RUN_ID}/"
)

Run : 20261012_005055
Dossier local : /Users/alexandro/Desktop/Master/Mines Paris/M2/Data Management & Cloud infrastructure for AI /de-platform-template/ingestion/audit_layout/20261012_005055
Layouts : 43
Fichiers Parquet : 1764
Volume total : 253.98 MiB
Destination : gs://data-engineering-a-alexandro/audit/partition_tests/20261012_005055/


## Upload on GCS 

In [15]:
from time import perf_counter

bucket = storage_client.bucket(BUCKET_NAME)

gcs_run_prefix = f"{AUDIT_PREFIX}/{RUN_ID}"

# Permet de reprendre un upload interrompu
existing_objects = {
    blob.name
    for blob in storage_client.list_blobs(
        BUCKET_NAME,
        prefix=f"{gcs_run_prefix}/"
    )
}

print("Objets déjà présents :", len(existing_objects))

upload_results = []

total_layouts = len(local_metrics_df)

for layout_number, row in enumerate( local_metrics_df.itertuples(index=False), start=1):

    table_name = row.table
    layout_name = row.layout
    local_layout_path = Path(row.local_path)

    files = sorted(
        local_layout_path.rglob("*.parquet")
    )

    layout_prefix = (
        f"{gcs_run_prefix}/"
        f"{table_name}/"
        f"{layout_name}"
    )

    print(
        f"\n[{layout_number}/{total_layouts}] "
        f"{table_name} / {layout_name}"
    )
    print(f"Fichiers à vérifier : {len(files):,}")

    start = perf_counter()

    uploaded_count = 0
    skipped_count = 0
    uploaded_bytes = 0

    progress_step = max(1, len(files) // 10)

    for file_number, local_file in enumerate(files, start=1):

        relative_file = local_file.relative_to(
            local_layout_path
        )

        object_name = (
            f"{layout_prefix}/"
            f"{relative_file.as_posix()}"
        )

        if object_name in existing_objects:
            skipped_count += 1
        else:
            blob = bucket.blob(object_name)

            blob.upload_from_filename(
                str(local_file),
                content_type="application/octet-stream",
            )

            uploaded_count += 1
            uploaded_bytes += local_file.stat().st_size
            existing_objects.add(object_name)

        if (
            file_number % progress_step == 0
            or file_number == len(files)
        ):
            progress_pct = (
                file_number / len(files) * 100
                if files
                else 100
            )

            print(
                f"  {file_number:,}/{len(files):,} "
                f"({progress_pct:.0f} %)"
            )

    upload_seconds = perf_counter() - start

    # Vérification réelle dans GCS
    gcs_blobs = list(
        storage_client.list_blobs(
            BUCKET_NAME,
            prefix=f"{layout_prefix}/"
        )
    )

    gcs_object_count = len(gcs_blobs)

    gcs_total_bytes = sum(
        blob.size or 0
        for blob in gcs_blobs
    )

    expected_object_count = len(files)
    expected_total_bytes = sum(
        file.stat().st_size
        for file in files
    )

    upload_valid = (
        gcs_object_count == expected_object_count
        and gcs_total_bytes == expected_total_bytes
    )

    upload_results.append({
        "run_id": RUN_ID,
        "table": table_name,
        "layout": layout_name,
        "gcs_prefix": f"gs://{BUCKET_NAME}/{layout_prefix}/",
        "uploaded_count": uploaded_count,
        "skipped_count": skipped_count,
        "upload_seconds": upload_seconds,
        "gcs_object_count": gcs_object_count,
        "gcs_total_bytes": gcs_total_bytes,
        "upload_valid": upload_valid,
    })

    print(
        f"Terminé : {uploaded_count:,} envoyé(s), "
        f"{skipped_count:,} déjà présent(s), "
        f"{upload_seconds:.2f} s"
    )

    print(
        f"Vérification GCS : "
        f"{gcs_object_count:,} objet(s), "
        f"{gcs_total_bytes:,} octets, "
        f"{'OK' if upload_valid else 'ERROR'}"
    )

upload_results_df = pd.DataFrame(upload_results)

Objets déjà présents : 0

[1/43] buyer / unpartitioned
Fichiers à vérifier : 1
  1/1 (100 %)
Terminé : 1 envoyé(s), 0 déjà présent(s), 1.66 s
Vérification GCS : 1 objet(s), 3,252,070 octets, OK

[2/43] carrier / unpartitioned
Fichiers à vérifier : 1
  1/1 (100 %)
Terminé : 1 envoyé(s), 0 déjà présent(s), 0.16 s
Vérification GCS : 1 objet(s), 2,768 octets, OK

[3/43] cart / unpartitioned
Fichiers à vérifier : 1
  1/1 (100 %)
Terminé : 1 envoyé(s), 0 déjà présent(s), 0.13 s
Vérification GCS : 1 objet(s), 2,944 octets, OK

[4/43] cart_items / unpartitioned
Fichiers à vérifier : 1
  1/1 (100 %)
Terminé : 1 envoyé(s), 0 déjà présent(s), 0.16 s
Vérification GCS : 1 objet(s), 2,366 octets, OK

[5/43] category / unpartitioned
Fichiers à vérifier : 1
  1/1 (100 %)
Terminé : 1 envoyé(s), 0 déjà présent(s), 0.12 s
Vérification GCS : 1 objet(s), 3,221 octets, OK

[6/43] category / by_category_id
Fichiers à vérifier : 2
  1/2 (50 %)
  2/2 (100 %)
Terminé : 2 envoyé(s), 0 déjà présent(s), 0.25 s
Vér

## BIG QUERIES

In [16]:
from google.cloud import bigquery
from google.api_core.exceptions import NotFound
import re

BIGQUERY_DATASET = "alexandro_dataset_bronze"
EXPECTED_LOCATION = "europe-west1"

dataset_id = f"{PROJECT_ID}.{BIGQUERY_DATASET}"

try:
    dataset = bigquery_client.get_dataset(dataset_id)

    print("Dataset trouvé :", dataset_id)
    print("Région :", dataset.location)

except NotFound:
    raise RuntimeError(
        f"Le dataset {dataset_id} n'existe pas."
    )

if dataset.location.lower() != EXPECTED_LOCATION.lower():
    raise RuntimeError(
        f"Mauvaise région : {dataset.location}. "
        f"Région attendue : {EXPECTED_LOCATION}."
    )

existing_tables = list(
    bigquery_client.list_tables(dataset_id)
)

print("Tables déjà présentes :", len(existing_tables))

for table in existing_tables:
    print("-", table.table_id)

Dataset trouvé : data-engineering-a.alexandro_dataset_bronze
Région : europe-west1
Tables déjà présentes : 14
- audit_20261011_210736__buyer__unpartitioned
- audit_20261011_210736__carrier__unpartitioned
- audit_20261011_210736__cart__unpartitioned
- audit_20261011_210736__cart_items__unpartitioned
- audit_20261011_210736__category__by_category_id
- audit_20261011_210736__category__unpartitioned
- audit_20261011_210736__customer__unpartitioned
- audit_20261011_210736__customer_payment__unpartitioned
- audit_20261011_210736__customer_shipping__by_is_default
- audit_20261011_210736__customer_shipping__unpartitioned
- audit_20261011_210736__daily_deals__unpartitioned
- month_by_day
- month_by_day_product
- order_items_by_day


In [18]:
STANDARD_STORAGE_USD_PER_GIB_MONTH = 0.020
CLASS_A_USD_PER_1000_OPERATIONS = 0.005
BYTES_PER_GIB = 1024 ** 3

final_metrics_df = local_metrics_df.merge(
    upload_results_df,
    on=["run_id", "table", "layout"],
    how="left",
    validate="one_to_one",
)

print("Lignes attendues :", len(local_metrics_df))
print("Lignes finales :", len(final_metrics_df))
print(
    "Uploads validés :",
    int(final_metrics_df["upload_valid"].sum()),
    "/",
    len(final_metrics_df),
)

final_metrics_df["storage_cost_usd_month"] = (
    final_metrics_df["gcs_total_bytes"]
    / BYTES_PER_GIB
    * STANDARD_STORAGE_USD_PER_GIB_MONTH
)

final_metrics_df["class_a_write_cost_usd"] = (
    final_metrics_df["gcs_object_count"]
    / 1000
    * CLASS_A_USD_PER_1000_OPERATIONS
)

Lignes attendues : 43
Lignes finales : 43
Uploads validés : 43 / 43


In [19]:
def safe_bigquery_name(value):
    value = re.sub(
        r"[^A-Za-z0-9_]",
        "_",
        value,
    )

    return value[:1024]


external_table_results = []

total_layouts = len(final_metrics_df)

for number, row in enumerate(
    final_metrics_df.itertuples(index=False),
    start=1,
):
    table_name = row.table
    layout_name = row.layout
    gcs_prefix = row.gcs_prefix.rstrip("/") + "/"

    # Le préfixe audit + RUN_ID évite les collisions
    bigquery_table_name = safe_bigquery_name(
        f"audit_{RUN_ID}__"
        f"{table_name}__"
        f"{layout_name}"
    )

    table_id = (
        f"{PROJECT_ID}."
        f"{BIGQUERY_DATASET}."
        f"{bigquery_table_name}"
    )

    print(
        f"[{number}/{total_layouts}] "
        f"{table_name} / {layout_name}"
    )

    external_config = bigquery.ExternalConfig(
        "PARQUET"
    )

    external_config.source_uris = [
        f"{gcs_prefix}*"
    ]

    external_config.autodetect = True

    if layout_name != "unpartitioned":

        hive_options = (
            bigquery.HivePartitioningOptions()
        )

        hive_options.mode = "STRINGS"
        hive_options.source_uri_prefix = gcs_prefix

        external_config.hive_partitioning = (
            hive_options
        )

    external_table = bigquery.Table(table_id)

    external_table.external_data_configuration = (
        external_config
    )

    try:
        created_table = (
            bigquery_client.create_table(
                external_table,
                exists_ok=False,
            )
        )

        creation_status = "CREATED"

    except Exception as error:

        # Autorise uniquement une table déjà créée
        # par ce même run
        try:
            existing_table = (
                bigquery_client.get_table(table_id)
            )

            creation_status = "ALREADY EXISTS"

        except Exception:
            raise error

    external_table_results.append({
        "run_id": RUN_ID,
        "table": table_name,
        "layout": layout_name,
        "bigquery_table_id": table_id,
        "creation_status": creation_status,
    })

    print(
        f"  {creation_status} : "
        f"{bigquery_table_name}"
    )

external_tables_df = pd.DataFrame(
    external_table_results
)

print("\nRésumé :")
print(
    external_tables_df[
        "creation_status"
    ].value_counts()
)

print(
    "Tables d’audit référencées :",
    len(external_tables_df),
)

[1/43] buyer / unpartitioned
  CREATED : audit_20261012_005055__buyer__unpartitioned
[2/43] carrier / unpartitioned
  CREATED : audit_20261012_005055__carrier__unpartitioned
[3/43] cart / unpartitioned
  CREATED : audit_20261012_005055__cart__unpartitioned
[4/43] cart_items / unpartitioned
  CREATED : audit_20261012_005055__cart_items__unpartitioned
[5/43] category / unpartitioned
  CREATED : audit_20261012_005055__category__unpartitioned
[6/43] category / by_category_id
  CREATED : audit_20261012_005055__category__by_category_id
[7/43] customer / unpartitioned
  CREATED : audit_20261012_005055__customer__unpartitioned
[8/43] customer_payment / unpartitioned
  CREATED : audit_20261012_005055__customer_payment__unpartitioned
[9/43] customer_shipping / unpartitioned
  CREATED : audit_20261012_005055__customer_shipping__unpartitioned
[10/43] customer_shipping / by_is_default
  CREATED : audit_20261012_005055__customer_shipping__by_is_default
[11/43] daily_deals / unpartitioned
  CREATED :

In [22]:
query_filters = {}

for table_name, candidates in partition_config.items():

    valid_candidates = [
        candidate
        for candidate in candidates
        if candidate is not None
    ]

    query_filters[table_name] = (
        max(valid_candidates, key=len)
        if valid_candidates
        else []
    )

def sql_literal(value, bigquery_type):

    if pd.isna(value):
        return "NULL"

    bigquery_type = bigquery_type.upper()

    if bigquery_type in {
        "INTEGER",
        "INT64",
        "FLOAT",
        "FLOAT64",
        "NUMERIC",
        "BIGNUMERIC",
    }:
        return str(value)

    if bigquery_type in {"BOOLEAN", "BOOL"}:
        return "TRUE" if bool(value) else "FALSE"

    safe_value = str(value).replace("'", "''")

    if bigquery_type == "DATE":
        return f"DATE '{safe_value[:10]}'"

    if bigquery_type == "DATETIME":
        return f"DATETIME '{safe_value}'"

    if bigquery_type == "TIMESTAMP":
        return f"TIMESTAMP '{safe_value}'"

    return f"'{safe_value}'"

In [23]:
bq_results = []

total_tests = len(external_tables_df)

for number, row in enumerate(
    external_tables_df.itertuples(index=False),
    start=1,
):
    table_name = row.table
    layout_name = row.layout
    table_id = row.bigquery_table_id

    print(
        f"\n[{number}/{total_tests}] "
        f"{table_name} / {layout_name}"
    )

    filter_columns = query_filters[table_name]

    bq_table = bigquery_client.get_table(table_id)

    schema_types = {
        field.name: field.field_type
        for field in bq_table.schema
    }

    if filter_columns:

        source_df = pd.read_csv(
            SOURCE_DIR / f"{table_name}.csv"
        )

        usable_rows = source_df.dropna(
            subset=filter_columns
        )

        if usable_rows.empty:
            where_clause = ""
        else:
            sample_row = usable_rows.iloc[0]

            conditions = []

            for column in filter_columns:

                literal = sql_literal(
                    sample_row[column],
                    schema_types[column],
                )

                conditions.append(
                    f"`{column}` = {literal}"
                )

            where_clause = (
                "\nWHERE "
                + "\n  AND ".join(conditions)
            )

    else:
        where_clause = ""

    sql = f"""
    SELECT COUNT(*) AS selected_rows
    FROM `{table_id}`
    {where_clause}
    """

    query_job = bigquery_client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            use_query_cache=False
        ),
        location=EXPECTED_LOCATION,
    )

    query_rows = list(query_job.result())

    elapsed_ms = (
        query_job.ended - query_job.started
    ).total_seconds() * 1000

    bytes_processed = (
        query_job.total_bytes_processed or 0
    )

    selected_rows = query_rows[0]["selected_rows"]

    bq_results.append({
        "run_id": RUN_ID,
        "table": table_name,
        "layout": layout_name,
        "query_elapsed_ms": elapsed_ms,
        "query_bytes_processed": bytes_processed,
        "selected_rows": selected_rows,
    })

    print("  Lignes retournées :", selected_rows)
    print("  Temps :", f"{elapsed_ms:.0f} ms")
    print("  Octets :", f"{bytes_processed:,}")

bq_metrics_df = pd.DataFrame(bq_results)


[1/43] buyer / unpartitioned
  Lignes retournées : 100000
  Temps : 184 ms
  Octets : 0

[2/43] carrier / unpartitioned
  Lignes retournées : 4
  Temps : 192 ms
  Octets : 0

[3/43] cart / unpartitioned
  Lignes retournées : 0
  Temps : 182 ms
  Octets : 0

[4/43] cart_items / unpartitioned
  Lignes retournées : 0
  Temps : 172 ms
  Octets : 0

[5/43] category / unpartitioned
  Lignes retournées : 1
  Temps : 266 ms
  Octets : 16

[6/43] category / by_category_id
  Lignes retournées : 1
  Temps : 227 ms
  Octets : 0

[7/43] customer / unpartitioned
  Lignes retournées : 130766
  Temps : 185 ms
  Octets : 0

[8/43] customer_payment / unpartitioned
  Lignes retournées : 100000
  Temps : 206 ms
  Octets : 0

[9/43] customer_shipping / unpartitioned
  Lignes retournées : 100000
  Temps : 279 ms
  Octets : 1,600,008

[10/43] customer_shipping / by_is_default
  Lignes retournées : 100000
  Temps : 244 ms
  Octets : 0

[11/43] daily_deals / unpartitioned
  Lignes retournées : 0
  Temps : 178

In [24]:

final_metrics_df = final_metrics_df.merge(
    bq_metrics_df,
    on=["run_id", "table", "layout"],
    how="left",
    validate="one_to_one",
)

final_columns = [
    "table",
    "layout",
    "class_a_write_cost_usd",
    "storage_cost_usd_month",
    "query_elapsed_ms",
    "query_bytes_processed",
]

final_results_df = final_metrics_df[
    final_columns
]

## Compute storage and writing costs

In [26]:
STANDARD_STORAGE_USD_PER_GIB_MONTH = 0.020
CLASS_A_USD_PER_1000_OPERATIONS = 0.005
BYTES_PER_GIB = 1024 ** 3

final_metrics_df = (
    local_metrics_df
    .merge(
        upload_results_df,
        on=["run_id", "table", "layout"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        bq_metrics_df,
        on=["run_id", "table", "layout"],
        how="left",
        validate="one_to_one",
    )
)

final_metrics_df["storage_cost_usd_month"] = (
    final_metrics_df["gcs_total_bytes"]
    / BYTES_PER_GIB
    * STANDARD_STORAGE_USD_PER_GIB_MONTH
)

final_metrics_df["class_a_write_cost_usd"] = (
    final_metrics_df["gcs_object_count"]
    / 1000
    * CLASS_A_USD_PER_1000_OPERATIONS
)

final_columns = [
    "table",
    "layout",
    "class_a_write_cost_usd",
    "storage_cost_usd_month",
    "query_elapsed_ms",
    "query_bytes_processed",
]

final_results_df = final_metrics_df[
    final_columns
]

print("Lignes finales :", len(final_results_df))
print("\nValeurs manquantes :")
print(final_results_df.isna().sum())

Lignes finales : 43

Valeurs manquantes :
table                     0
layout                    0
class_a_write_cost_usd    0
storage_cost_usd_month    0
query_elapsed_ms          0
query_bytes_processed     0
dtype: int64


In [27]:
metrics_columns = [
    "class_a_write_cost_usd",
    "storage_cost_usd_month",
    "query_elapsed_ms",
    "query_bytes_processed",
]

display_columns = [
    "table",
    "layout",
    *metrics_columns,
]

green_style = {
    "background-color": "#b7e4c7",
    "color": "#081c15",
    "font-weight": "bold",
}

red_style = {
    "background-color": "#ffadad",
    "color": "#450920",
    "font-weight": "bold",
}

styled_comparison = (
    final_results_df[display_columns]
    .sort_values(["table", "layout"])
    .reset_index(drop=True)
)

for table_name in styled_comparison["table"].unique():

    table_df = (
        styled_comparison[
            styled_comparison["table"] == table_name
        ]
        .reset_index(drop=True)
    )

    print(f"\n{table_name}")

    styled_table = table_df.style

    for metric in metrics_columns:

        minimum = table_df[metric].min()
        maximum = table_df[metric].max()

        if minimum == maximum:

            styled_table = styled_table.set_properties(
                subset=pd.IndexSlice[
                    table_df.index.tolist(),
                    [metric],
                ],
                **green_style,
            )

        else:

            best_indices = table_df.index[
                table_df[metric] == minimum
            ].tolist()

            worst_indices = table_df.index[
                table_df[metric] == maximum
            ].tolist()

            styled_table = styled_table.set_properties(
                subset=pd.IndexSlice[
                    best_indices,
                    [metric],
                ],
                **green_style,
            )

            styled_table = styled_table.set_properties(
                subset=pd.IndexSlice[
                    worst_indices,
                    [metric],
                ],
                **red_style,
            )

    display(
        styled_table.format({
            "class_a_write_cost_usd": "${:.12e}",
            "storage_cost_usd_month": "${:.12e}",
            "query_elapsed_ms": "{:,.0f} ms",
            "query_bytes_processed": "{:,.0f} bytes",
        })
    )


buyer


,table,layout,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,buyer,unpartitioned,$5.000000000000e-06,$6.057452410460e-05,184 ms,0 bytes



carrier


,table,layout,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,carrier,unpartitioned,$5.000000000000e-06,$5.155801773071e-08,192 ms,0 bytes



cart


...

,table,layout,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,shipping_details,by_country,$3.000000000000e-05,$1.770767383277e-04,229 ms,"843,946 bytes"
1,shipping_details,by_country_then_state,$4.150000000000e-04,$1.987316086888e-04,119 ms,0 bytes
2,shipping_details,by_state,$4.150000000000e-04,$2.014139294624e-04,305 ms,0 bytes
3,shipping_details,unpartitioned,$5.000000000000e-06,$1.595568843186e-04,287 ms,"4,423,446 bytes"



subscription


,table,layout,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,subscription,unpartitioned,$5.000000000000e-06,$3.881709650159e-05,189 ms,0 bytes



wishlist_item


,table,layout,class_a_write_cost_usd,storage_cost_usd_month,query_elapsed_ms,query_bytes_processed
0,wishlist_item,unpartitioned,$5.000000000000e-06,$1.759639009833e-05,157 ms,0 bytes
